# Technical Support Agent
**ResolveX - M5: Ticket analysis and resolution**

Qualitative evaluation of a prototype that combines the saved M3 classifier, frozen M4 Ask Ubuntu retriever, curated Ubuntu documentation and Gemini.

## 1. Configure Gemini

The API key is read from the local `.env` file and is not displayed.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "src/support_agent.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook within ResolveX")
load_dotenv(ROOT / ".env")
if not os.getenv("GEMINI_API_KEY", "").strip():
    raise ValueError("GEMINI_API_KEY is missing from the project .env file")
print("Gemini API key is configured.")

Gemini API key is configured.


## 2. Load the support workflow

Load the saved classifier and frozen M4 retriever. Build or load the separate MiniLM + FAISS index for selected Ubuntu documentation. Retrieved evidence keeps its source type, URL and release scope.

In [2]:
from pathlib import Path
import sys
import os
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "src/support_agent.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook within ResolveX")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.support_agent import load_support_agent

load_dotenv(ROOT / ".env")
REFRESH_OFFICIAL_DOCS = False  # Set True only to refresh the local documentation snapshot.
agent = load_support_agent(ROOT, os.environ["GEMINI_API_KEY"].strip(),
                           refresh_official_docs=REFRESH_OFFICIAL_DOCS)
doc_manifest = agent.official_retriever.manifest
print("Ask Ubuntu retriever: loaded from the frozen M4 artifacts")
print("Official documentation pages indexed:", doc_manifest.get("source_count"))
print("Official documentation chunks:", doc_manifest.get("chunk_count"))
print("Failed official pages:")
if doc_manifest.get("failed_sources"):
    display(pd.DataFrame(doc_manifest["failed_sources"]))
else:
    print("None")
print("Official documentation coverage by category:")
indexed_categories = sorted({category for chunk in agent.official_retriever.chunks for category in chunk["categories"]})
all_categories = sorted(agent.classifier["pipeline"].classes_)
for category in all_categories:
    print(f"- {category}: {'available' if category in indexed_categories else 'no indexed page'}")
print("Category coverage describes the index metadata; it does not guarantee relevance to each ticket.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ask Ubuntu retriever: loaded from the frozen M4 artifacts
Official documentation pages indexed: 12
Official documentation chunks: 304
Failed official pages:
None
Official documentation coverage by category:
- Access & Permissions: available
- Boot & Startup: available
- Hardware & Drivers: available
- Network & Connectivity: available
- Software & Packages: available
- Storage & Filesystems: available
Category coverage describes the index metadata; it does not guarantee relevance to each ticket.


In [3]:
from src.support_agent import (
    required_ticket_details,
    _normalized_question_text,
    _near_duplicate_similarity,
    _unsupported_nvidia_commands,
    _draft_action_safety_issues,
    _official_release_compatibility_issues,
    _select_official_hits,
    _allow_read_only_diagnostic,
)

assert required_ticket_details(
    "Install a desktop application",
    "On Ubuntu Desktop 26.04, I want to install a normal application from the Ubuntu App Center.",
    "Software & Packages",
) == []
assert required_ticket_details(
    "Why does apt-get install update fail?",
    "I ran sudo apt-get install update and got E: Unable to locate package update. I meant to update the installed software.",
    "Software & Packages",
) == []
assert required_ticket_details(
    "APT reports held broken packages",
    "Ubuntu 24.04 cannot install an application and reports held broken packages. I do not have the complete terminal output.",
    "Software & Packages",
) == ["complete package-manager error message"]
assert _unsupported_nvidia_commands(
    {"steps": [{"instruction": "Run sudo ubuntu-drivers install nvidia:535"}]},
    "Ubuntu Desktop 26.04 NVIDIA driver is missing",
)
release_draft = {"steps": [{"instruction": "Use the Steam launch option", "citations": [{"source_id": "OD1", "quote": "example quote"}]}]}
release_ticket = "Kubuntu 21.10 does not use the NVIDIA GPU for Steam games."
unscoped_source = [{"source_id": "OD1", "source_type": "official_ubuntu_documentation", "release_scope": "The page does not state a specific Ubuntu release"}]
matched_source = [{"source_id": "OD1", "source_type": "official_ubuntu_documentation", "release_scope": "Ubuntu Desktop 21.10"}]
assert _official_release_compatibility_issues(release_draft, unscoped_source, release_ticket)
assert not _official_release_compatibility_issues(release_draft, matched_source, release_ticket)
assert _draft_action_safety_issues({"steps": [{"instruction": "sudo mount /dev/<driveID>1 /mnt"}]})
assert _draft_action_safety_issues({"steps": [{"instruction": "sudo chmod 0777 /mnt"}]})
assert not _draft_action_safety_issues({"steps": [{"instruction": "Run nmcli device status and review the output"}]})
assert not _unsupported_nvidia_commands(
    {"steps": [{"instruction": "Run sudo ubuntu-drivers install --recommended"}]},
    "Ubuntu Desktop 26.04 NVIDIA driver is missing",
)
assert _unsupported_nvidia_commands(
    {"steps": [{"instruction": "sudo apt install linux-modules-nvidia-${DRIVER_BRANCH}-${LINUX_FLAVOUR}"}]},
    "Ubuntu Desktop 26.04 NVIDIA driver is missing",
)

source_candidates = [
    {"title": "Ubuntu manpage: nmcli", "categories": ["Network & Connectivity"], "release_scope": "command reference"},
    {"title": "Networking", "categories": ["Network & Connectivity"], "release_scope": "current documentation"},
    {"title": "Bluetooth", "categories": ["Hardware & Drivers"], "release_scope": "Ubuntu Desktop 26.04"},
    {"title": "Ubuntu 26.04 LTS NVIDIA driver guidance", "categories": ["Hardware & Drivers"], "release_scope": "Ubuntu 26.04 LTS"},
    {"title": "Install Ubuntu Desktop", "categories": ["Boot & Startup", "Storage & Filesystems"], "release_scope": "Ubuntu Desktop 26.04; installation only"},
    {"title": "Ubuntu manpage: ls", "categories": ["Access & Permissions"], "release_scope": "command reference"},
]
network_sources = _select_official_hits(
    source_candidates, "Ubuntu laptop cannot connect to Wi-Fi; other devices connect to the same network.",
    "Network & Connectivity", k=6,
)
assert {source["title"] for source in network_sources} == {"Ubuntu manpage: nmcli", "Networking"}
bluetooth_sources = _select_official_hits(
    source_candidates, "Bluetooth adapter firmware error", "Network & Connectivity", k=6,
)
assert "Bluetooth" in {source["title"] for source in bluetooth_sources}
boot_sources = _select_official_hits(
    source_candidates, "Ubuntu fails to boot after partition changes", "Boot & Startup", k=6,
)
assert not boot_sources
permission_sources = _select_official_hits(
    source_candidates, "Cannot inspect a document's owner in my home folder", "Access & Permissions", k=6,
)
assert [source["title"] for source in permission_sources] == ["Ubuntu manpage: ls"]

diagnostic_source = {"source_id": "OD1", "source_type": "official_ubuntu_manpage"}
diagnostic_draft = {"steps": [{"instruction": "Run `nmcli device status` and review the output.", "citations": [{"source_id": "OD1", "quote": "supported excerpt"}]}]}
diagnostic_context = {"weak_evidence": False, "category_source_mismatch": False, "sources": [diagnostic_source]}
assert _allow_read_only_diagnostic(diagnostic_draft, diagnostic_context)
assert not _allow_read_only_diagnostic(diagnostic_draft, {**diagnostic_context, "weak_evidence": True})
assert not _allow_read_only_diagnostic(diagnostic_draft, {**diagnostic_context, "category_source_mismatch": True})
community_context = {**diagnostic_context, "sources": [{"source_id": "S1", "source_type": "askubuntu_community_case"}]}
community_draft = {"steps": [{"instruction": "Run `nmcli device status`.", "citations": [{"source_id": "S1", "quote": "supported excerpt"}]}]}
assert not _allow_read_only_diagnostic(community_draft, community_context)
assert not _allow_read_only_diagnostic({"steps": [{"instruction": "Run `sudo apt upgrade`.", "citations": [{"source_id": "OD1", "quote": "supported excerpt"}]}]}, diagnostic_context)

assert _normalized_question_text("Wi-Fi issue", "Can't connect!") == "wi fi issue can t connect"
assert _near_duplicate_similarity("one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty", "one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty-one") >= 0.92
print("Deterministic retrieval selection and safe-diagnostic checks passed.")


Deterministic retrieval selection and safe-diagnostic checks passed.


## 3. Prepare the evaluation cases

The 12 cases cover two examples per project category. H1, S1 and T1 use real Ask Ubuntu questions; only their titles and bodies enter the agent. Their original answers and recorded duplicate groups are excluded from retrieval. Expected behavior is set before generation.

In [4]:
import sqlite3
from collections import defaultdict, deque

# H1, S1 and T1 are real Ask Ubuntu questions. Only question text is passed to the agent.
# Each source question and its recorded duplicate-connected group are withheld from retrieval.
source_ids = {"H1": "1411564", "S1": "388541", "T1": "1500793"}
source_categories = {"H1": "Hardware & Drivers", "S1": "Software & Packages", "T1": "Storage & Filesystems"}

db_uri = (ROOT / "data/processed/retrieval/evidence.sqlite").as_uri() + "?mode=ro"
edges = pd.read_csv(ROOT / "data/processed/duplicate_relationships.csv", dtype=str)
adjacency = defaultdict(set)
for question_id, related_id in edges[["QuestionId", "RelatedQuestionId"]].itertuples(index=False, name=None):
    adjacency[question_id].add(related_id)
    adjacency[related_id].add(question_id)

def duplicate_component(question_id):
    found, pending = {question_id}, deque([question_id])
    while pending:
        current = pending.popleft()
        for neighbor in adjacency.get(current, ()):
            if neighbor not in found:
                found.add(neighbor)
                pending.append(neighbor)
    return sorted(found, key=int)

def load_source_question(case_id, expected):
    question_id = source_ids[case_id]
    with sqlite3.connect(db_uri, uri=True) as connection:
        row = connection.execute(
            "SELECT title, question_text FROM cases WHERE question_id = ?", (question_id,)
        ).fetchone()
    if row is None:
        raise ValueError(f"Ask Ubuntu source question {question_id} is missing from the M4 corpus")
    return {
        "id": case_id, "category": source_categories[case_id],
        "expected": expected, "title": row[0], "body": row[1],
        "source_question_id": question_id,
        "source_url": f"https://askubuntu.com/questions/{question_id}",
        "exclude_question_ids": duplicate_component(question_id),
    }

s1 = load_source_question(
    "S1",
    "The official Ubuntu package-management guide distinguishes `apt update`, which refreshes the package index, from `apt upgrade`, which installs available upgrades. Explain why `apt-get install update` treats `update` as a package name, give the documented command for the user's stated intent, and describe the separate upgrade action without automatic confirmation. Do not infer package-repair steps.",
)
h1 = load_source_question(
    "H1",
    "The Ubuntu Steam guide directly addresses selecting the NVIDIA GPU for a game. Check its full instructions against the reported Kubuntu 21.10 setup; since the page does not state release compatibility and 21.10 is obsolete, disclose this limitation and request engineer review rather than presenting the launch option as verified for this system. Do not guess driver changes.",
)
t1 = load_source_question(
    "T1",
    "Needs review; the gap between filesystem and visible-directory usage has multiple possible causes. Do not suggest deleting data or changing filesystem settings without identifying the cause.",
)

evaluation_cases = [
    {"id":"N1", "category":"Network & Connectivity", "expected":"The Ubuntu 26.04 `nmcli` manpage supports the read-only `nmcli device status` check to record interface and connection state. Use that output to guide the next diagnostic; do not change network configuration before reviewing it.", "title":"Ubuntu laptop cannot connect to Wi-Fi", "body":"Ubuntu Desktop 26.04 sees my home Wi-Fi network, but it will not connect after I enter the password. Other devices connect to the same network."},
    {"id":"N2", "category":"Network & Connectivity", "expected":"Needs review; do not change network configuration without the interface, topology and error details.", "title":"Server lost network access after a change", "body":"An Ubuntu Server host stopped reaching the office network after a network configuration change. I do not know which interface or gateway was changed."},
    {"id":"B1", "category":"Boot & Startup", "expected":"Needs review; do not prescribe bootloader or partition repair without exact symptoms and layout.", "title":"Ubuntu fails to boot after partition changes", "body":"After resizing partitions, Ubuntu 24.04 shows a GRUB error and does not reach the login screen. Windows is also installed on this computer."},
    {"id":"B2", "category":"Boot & Startup", "expected":"Needs review when release-matched repair evidence is missing; installation instructions are not boot-repair steps.", "title":"Ubuntu Server hangs during startup", "body":"Ubuntu Server 24.04 pauses during startup after a recent update. I have not captured the last messages or confirmed whether recovery mode is available."},
    {"id":"S2", "category":"Software & Packages", "expected":"Needs review; ask for the complete package-manager error and repository context before suggesting repair commands.", "title":"APT reports held broken packages", "body":"Ubuntu 24.04 cannot install an application and reports held broken packages. I do not have the complete terminal output."},
    {"id":"H2", "category":"Hardware & Drivers", "expected":"Needs review when the adapter or firmware error is unknown; avoid arbitrary vendor downloads and firmware changes.", "title":"Bluetooth adapter firmware error", "body":"Ubuntu 26.04 reports a Bluetooth firmware load error, but I do not know the adapter model or the firmware path shown in the full log."},
    {"id":"T2", "category":"Storage & Filesystems", "expected":"Needs review; do not run filesystem repair or partition-changing tools without the device, filesystem, error and backup status.", "title":"External disk may have filesystem damage", "body":"Ubuntu says an external disk may be damaged. I have not confirmed the device name, filesystem, exact error or whether its data is backed up."},
    {"id":"P1", "category":"Access & Permissions", "expected":"Give a targeted, low-risk permissions step only if the file path, owner and requested access are clear; otherwise ask for those details.", "title":"Cannot edit a personal document", "body":"On Ubuntu Desktop 26.04, I cannot edit a document in my home folder. I want to inspect its owner and permissions before changing anything."},
    {"id":"P2", "category":"Access & Permissions", "expected":"Needs review; do not recommend recursive chmod/chown or broad privileged changes without path, ownership and impact details.", "title":"Application cannot access a system directory", "body":"An application reports permission denied for a directory under /var. I do not know which process owns it or whether the directory is managed by a package."},
]
evaluation_cases.insert(4, s1)
evaluation_cases.insert(7, h1)
evaluation_cases.insert(9, t1)

assert len(evaluation_cases) == 12
assert {case["category"] for case in evaluation_cases} == set(agent.classifier["pipeline"].classes_)
for case_id, question_id in source_ids.items():
    case = next(case for case in evaluation_cases if case["id"] == case_id)
    assert question_id in case["exclude_question_ids"]
print("Cases:", len(evaluation_cases), "| two per category | expected behavior set before model calls")
for case in (h1, s1, t1):
    print(case["id"], case["source_url"], "| source group IDs excluded:", len(case["exclude_question_ids"]))
print("For these cases, only question text is loaded; original answers are not passed to the agent.")


Cases: 12 | two per category | expected behavior set before model calls
H1 https://askubuntu.com/questions/1411564 | source group IDs excluded: 1
S1 https://askubuntu.com/questions/388541 | source group IDs excluded: 1
T1 https://askubuntu.com/questions/1500793 | source group IDs excluded: 2
For these cases, only question text is loaded; original answers are not passed to the agent.


## 4. Generate and review recommendations

The agent drafts recommendations from retrieved evidence. Check each citation against its source, then review relevance, technical correctness, release fit and command risk. A `recommendation` status or quote match alone is not a pass.

In [5]:
# One Gemini request is sent for each of the 12 predefined cases.
evaluation_results = []
for case in evaluation_cases:
    response = agent.run(
        case["title"], case["body"],
        exclude_question_ids=case.get("exclude_question_ids"),
    )
    evaluation_results.append({"case": case, "response": response})
    print(case["id"], response.get("status"), "steps:", len(response.get("steps", [])),
          "review:", response.get("engineer_review_required"), "error:", response.get("error"))

summary_rows = []
for item in evaluation_results:
    case, response = item["case"], item["response"]
    source_by_id = {source["source_id"]: source for source in response.get("sources", [])}
    cited_types = sorted({
        source_by_id[citation["source_id"]]["source_type"]
        for step in response.get("steps", []) for citation in step["citations"]
        if citation.get("source_id") in source_by_id
    })
    audit = response.get("retrieval_audit") or {}
    summary_rows.append({
        "Case": case["id"], "Expected category": case["category"],
        "Predicted category": response.get("category"),
        "Classifier review": response.get("classification", {}).get("human_review"),
        "Expected behavior": case["expected"], "Status": response.get("status"),
        "Steps": len(response.get("steps", [])), "Cited source types": ", ".join(cited_types),
        "Citation check": response.get("citation_check"),
        "Missing information": "; ".join(response.get("missing_information", [])),
        "Review reasons": "; ".join(response.get("review_reasons", [])),
        "Known group IDs": len(audit.get("excluded_question_ids", [])),
        "Group hits removed": len(audit.get("excluded_question_hits", [])),
        "Exact/near duplicates removed": len(audit.get("exact_match_question_ids", [])) + len(audit.get("near_duplicate_question_ids", [])),
        "Retrieved Ask Ubuntu IDs": ", ".join(audit.get("retained_question_ids", [])),
        "Manual technical review": "Pending",
    })

with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 0):
    display(pd.DataFrame(summary_rows))

for item in evaluation_results:
    case, response = item["case"], item["response"]
    source_by_id = {source["source_id"]: source for source in response.get("sources", [])}
    print(f"\n--- {case['id']} | {response.get('status')} ---")
    if case.get("source_url"):
        print("Evaluation source (question only):", case["source_url"])
    if response.get("steps"):
        for number, step in enumerate(response["steps"], 1):
            print(f"Step {number}: {step['instruction']}")
            for citation in step["citations"]:
                source = source_by_id[citation["source_id"]]
                print(f"  Quote [{citation['source_id']}]: {citation['quote']}")
                print(f"  Source: {source.get('title')} | {source.get('url')} | {source.get('release_scope', 'release not stated')}")
    else:
        print("No troubleshooting steps provided.")
    if response.get("sources"):
        print("Retrieved sources:")
        for source in response["sources"]:
            print(f"  {source['source_id']} [{source['source_type']}] {source['title']} | {source.get('url')} | {source.get('release_scope', 'release not stated')}")
    if response.get("retrieval_audit"):
        print("Retrieval leakage audit:", response["retrieval_audit"])


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


N1 needs_review steps: 0 review: True error: None
N2 needs_review steps: 0 review: True error: None
B1 needs_review steps: 0 review: True error: None
B2 needs_review steps: 0 review: True error: None
S1 recommendation steps: 2 review: True error: None
S2 needs_review steps: 0 review: True error: None
H2 needs_review steps: 0 review: True error: None
H1 recommendation steps: 1 review: True error: None
T2 needs_review steps: 0 review: True error: None
T1 needs_review steps: 0 review: True error: None
P1 needs_review steps: 0 review: True error: None
P2 needs_review steps: 0 review: True error: None


,Case,Expected category,Predicted category,Classifier review,Expected behavior,Status,Steps,Cited source types,Citation check,Missing information,Review reasons,Known group IDs,Group hits removed,Exact/near duplicates removed,Retrieved Ask Ubuntu IDs,Manual technical review
0,N1,Network & Connectivity,Network & Connectivity,False,The Ubuntu 26.04 `nmcli` manpage supports the read-only `nmcli device status` check to record interface and connection state. Use that output to guide the next diagnostic; do not change network configuration before reviewing it.,needs_review,0,,no_steps,Wi-Fi interface name; System log or NetworkManager journal output during connection attempt,No directly relevant official Ubuntu documentation or verified official fix is present in the provided sources for authentication failures during Wi-Fi connection.; Community sources present older or generalized workarounds rather than verified official troubleshooting for Ubuntu 26.04.,0,0,0,,Pending
1,N2,Network & Connectivity,Network & Connectivity,False,"Needs review; do not change network configuration without the interface, topology and error details.",needs_review,0,,no_steps,Ubuntu release version; Netplan configuration files in /etc/netplan/; Current output of network status tools such as ip addr or ip route,No official step-by-step repair guide is available in the retrieved official documentation for resolving a post-configuration network outage without specific system details.,0,0,0,,Pending
2,B1,Boot & Startup,Boot & Startup,False,Needs review; do not prescribe bootloader or partition repair without exact symptoms and layout.,needs_review,0,,not_run,,The predicted category does not match the category of the top official-documentation result; confirm routing before troubleshooting.,0,0,0,,Pending
3,B2,Boot & Startup,Boot & Startup,False,Needs review when release-matched repair evidence is missing; installation instructions are not boot-repair steps.,needs_review,0,,no_steps,last visible startup messages or systemd logs from journalctl; confirmation of whether recovery mode is accessible,"Retrieved sources are community answers from older Ubuntu versions (20.04/11.04) addressing different hardware or scenarios (Dell laptops, Intel NUC, installation issues), which do not provide verified official guidance or safe actionable troubleshooting steps for Ubuntu Server 24.04.",0,0,0,,Pending
4,S1,Software & Packages,Software & Packages,False,"The official Ubuntu package-management guide distinguishes `apt update`, which refreshes the package index, from `apt upgrade`, which installs available upgrades. Explain why `apt-get install update` treats `update` as a package name, give the documented command for the user's stated intent, and describe the separate upgrade action without automatic confirmation. Do not infer package-repair steps.",recommendation,2,official_ubuntu_documentation,passed,,,1,0,0,"229369, 984534, 165676",Pending
5,S2,Software & Packages,Software & Packages,False,Needs review; ask for the complete package-manager error and repository context before suggesting repair commands.,needs_review,0,,not_run,complete package-manager error message,Package-repair commands require the Ubuntu version and complete error details.,0,0,0,,Pending
6,H2,Hardware & Drivers,Network & Connectivity,False,Needs review when the adapter or firmware error is unknown; avoid arbitrary vendor downloads and firmware changes.,needs_review,0,,not_run,,The predicted category does not match the category of the top official-documentation result; confirm routing before troubleshooting.,0,0,0,,Pending
7,H1,Hardware & Drivers,Hardware & Drivers,False,"The Ubuntu Steam guide directly addresses selecting the NVIDIA GPU for a game. Check its full instructions against the reported Kubuntu 21.10 setup; since the page does not state release compatibility and 21.10 is obsolete, disclose this limitation and request engineer review rather than presenting the launch option as verified for t


--- N1 | needs_review ---
No troubleshooting steps provided.
Retrieved sources:
  OD1 [official_ubuntu_manpage] Ubuntu manpage: nmcli | https://manpages.ubuntu.com/manpages/resolute/man1/nmcli.1.html | Ubuntu 26.04 LTS (Resolute); NetworkManager command reference
  OD2 [official_ubuntu_documentation] Networking | https://ubuntu.com/server/docs/how-to/networking/ | Ubuntu Server documentation; current documentation site
  S3 [askubuntu_community_case] WiFi repeatedly asks for password (Ubuntu 12.10 fresh) | https://askubuntu.com/a/211116 | release not stated
  S4 [askubuntu_community_case] Ubuntu Wifi keeps reconnecting even after router reconfiguring | https://askubuntu.com/a/1272915 | release not stated
  S5 [askubuntu_community_case] Unable to connect to any wifi network detected in Xubuntu | https://askubuntu.com/a/1362570 | release not stated

--- N2 | needs_review ---
No troubleshooting steps provided.
Retrieved sources:
  OD1 [official_ubuntu_manpage] Ubuntu manpage: nmcli | htt

### Qualitative review

| Case | Review | Result |
| --- | --- | --- |
| S1: APT command question | The official guide supports `apt update` and `apt upgrade`. | Partial: the response did not explain why `apt-get install update` treats `update` as a package name, so it does not meet the expected behavior. |
| H1: Steam on Kubuntu 21.10 | The guide is relevant, but it does not state release compatibility; the generated citation failed validation. | No steps returned. Withholding the draft was appropriate. |
| N1: Wi-Fi connection | The retrieved `nmcli` reference supports a read-only status check, but unrelated pages were also passed to the model. | The agent abstained instead of giving the expected low-risk diagnostic. |
| Other cases | Results include unrelated evidence, missing ticket details, or both. | Review each case below to separate retrieval problems from appropriate abstention. |

The package guide supports the two APT commands, but command support alone does not mean the response answered the ticket. The release gate also passed an offline integration check; H1's latest run was withheld by citation validation, so it does not show that gate firing.


## Case-by-case failure review

| Case | Main cause | Assessment |
| --- | --- | --- |
| N1 Wi-Fi | Relevant `nmcli` reference was retrieved, but the agent abstained; other retrieved pages were unrelated. | Missed a low-risk diagnostic. |
| N2 Network change | Networking guidance was present, but interface, topology and error details were missing; some results were unrelated. | Abstention was appropriate; retrieval precision needs work. |
| B1 GRUB after resize | Official results were unrelated; community cases were only adjacent to the problem. | Abstention was appropriate for a high-risk repair; retrieval failed to find direct evidence. |
| B2 Startup hang | No relevant release-matched startup guidance; logs and recovery details were missing. | Abstention was appropriate; retrieval coverage was weak. |
| S1 APT command | Official package guidance was relevant, but the draft did not explain the command error and added an upgrade action. | Partial generation; does not pass the expected behavior. |
| S2 Held packages | The full package error was missing, so the pre-generation detail check stopped the draft; retrieved evidence was broad. | Abstention was appropriate; request the error before repair advice. |
| H2 Bluetooth firmware | The classifier predicted Network; the Bluetooth page exposed the category conflict and the workflow stopped for review. | Review was appropriate with missing adapter/log details; routing conflict identified, M3 stays unchanged. |
| H1 Steam GPU | Relevant guide lacked stated compatibility with Kubuntu 21.10; generated quote failed validation. | Withholding was appropriate; citation generation failed. |
| T2 External disk | Generic device references did not support repair; device, filesystem, error and backup status were missing. | Abstention was appropriate; retrieved evidence was not specific enough. |
| T1 Disk usage | Some related references were retrieved, but the generated citation failed validation. | Safe withholding; evidence-grounded diagnostic generation needs review. |
| P1 File permissions | `ls` reference appeared among unrelated results; the ticket did not give the exact file path, but the response did not clearly request it. | Information request was incomplete; ask for the file path before giving a targeted check. |
| P2 System directory | File path/process details were missing and retrieved pages were mostly unrelated. | Abstention was appropriate; retrieval needs better relevance. |

The pilot exposed both retrieval and response-policy issues. For the next run, official context is now filtered by category/topic and the only diagnostic allowed alongside review is an explicitly cited `nmcli device status` command. A `needs_review` result is not automatically a failure: several cases lacked details or involved risky changes. N1 clearly missed an available safe check. P1 should have asked for the exact file path; its ticket did not specify one. S1 is only a partial answer. Do not count citation matching as technical validation.


## Findings and limitations

The 12-case pilot ran without notebook errors. The official index contained 12 pages and 304 chunks across all six categories. Eleven cases returned no troubleshooting steps; S1 returned one cited APT draft, but it did not meet the expected behavior. The case-by-case review of the previous run identified retrieval noise, a missed safe diagnostic, an incomplete missing-information request, and a category conflict, and a citation-generation failure.

- This small qualitative pilot does not estimate reliability on company tickets.
- The official documentation set is curated and incomplete; some sources do not state release compatibility.
- Community answers are evidence of similar cases, not verified solutions.
- Citation checks verify source traceability, not technical correctness. Engineers must review every recommendation.
- The next run applies category/topic filtering to official evidence and permits only the directly cited read-only `nmcli device status` check alongside review. M5 remains in progress until the updated 12-case results are manually reviewed.
